In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ✅ Edit this if your processed file lives elsewhere
GEO_VTD_PATH = Path("data/processed/geo_vtd.parquet")

# Output files (optional)
OUT_CSV = Path("texas_demographics_table.csv")
OUT_TEX = Path("texas_demographics_table.tex")

TITLE = "Texas demographics"


In [ ]:
geo = pd.read_parquet(GEO_VTD_PATH)
geo.columns = [c.strip() for c in geo.columns]
geo.head()

In [ ]:
def pick_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def sum_col(df, col):
    return int(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())

def fmt_pct(x):
    return f"{x*100:.2f}%"

GROUPS = [
    ("Latino", "hisp"),
    ("Black", "black"),
    ("White", "white"),
    ("Other", "other"),
]

# ---- column detection (edit here if your naming differs) ----

# total population
total_total_col = pick_existing(geo, ["total_pop", "tot_pop", "pop_total", "total"])

total_by_group = {
    "hisp": pick_existing(geo, ["total_hisp", "pop_hisp", "hisp_total", "tot_hisp"]),
    "black": pick_existing(geo, ["total_black", "pop_black", "black_total", "tot_black"]),
    "white": pick_existing(geo, ["total_white", "pop_white", "white_total", "tot_white"]),
    "other": pick_existing(geo, ["total_other", "pop_other", "other_total", "tot_other"]),
}

# VAP
vap_total_col = pick_existing(geo, ["vap_total", "vap"])
vap_by_group = {
    "hisp": pick_existing(geo, ["vap_hisp", "hispvap", "vap_latino"]),
    "black": pick_existing(geo, ["vap_black", "blackvap", "vap_nh_black", "vap_black_nh"]),
    "white": pick_existing(geo, ["vap_white", "whitevap", "vap_nh_white", "vap_white_nh"]),
    "other": pick_existing(geo, ["vap_other", "othervap"]),
}

# CVAP
cvap_total_col = pick_existing(geo, ["cvap_total", "cvap"])
cvap_by_group = {
    "hisp": pick_existing(geo, ["cvap_hisp", "cvap_latino"]),
    "black": pick_existing(geo, ["cvap_nh_black", "cvap_black", "cvap_black_nh"]),
    "white": pick_existing(geo, ["cvap_nh_white", "cvap_white", "cvap_white_nh"]),
    "other": pick_existing(geo, ["cvap_other"]),
}

print("Detected:")
print(" total_total_col:", total_total_col)
print(" vap_total_col  :", vap_total_col)
print(" cvap_total_col :", cvap_total_col)
print(" total_by_group :", total_by_group)
print(" vap_by_group   :", vap_by_group)
print(" cvap_by_group  :", cvap_by_group)


In [ ]:
if cvap_total_col is None:
    raise ValueError("cvap_total not found. Your geo_vtd.parquet must contain cvap_total (or cvap).")

total_count = sum_col(geo, total_total_col) if total_total_col else None
vap_count   = sum_col(geo, vap_total_col) if vap_total_col else None
cvap_count  = sum_col(geo, cvap_total_col)

def compute_group_totals(kind_total_col, kind_map, kind_name):
    """Return (group_totals_dict, denominator_total) or (None, None) if total missing."""
    if kind_total_col is None:
        return None, None

    denom = sum_col(geo, kind_total_col)

    totals = {}
    missing = []
    for _, key in GROUPS:
        c = kind_map.get(key)
        if c is None:
            missing.append(key)
        else:
            totals[key] = sum_col(geo, c)

    # If only 'other' missing, compute residual when possible
    if "other" in missing and all(kind_map.get(k) is not None for k in ["hisp", "black", "white"]):
        totals["other"] = max(0, denom - totals["hisp"] - totals["black"] - totals["white"])
        missing = [m for m in missing if m != "other"]

    if missing:
        print(f"[WARN] Missing {kind_name} by-group columns for: {missing}. Leaving those shares blank.")
    return totals, denom

total_grp, total_den = compute_group_totals(total_total_col, total_by_group, "Total population")
vap_grp, vap_den     = compute_group_totals(vap_total_col, vap_by_group, "VAP")
cvap_grp, cvap_den   = compute_group_totals(cvap_total_col, cvap_by_group, "CVAP")

rows = []
for group_name, key in GROUPS:
    row = {"Racial group": group_name}

    row["Share of total population"] = fmt_pct(total_grp[key] / total_den) if total_grp and total_den and key in total_grp else ""
    row["Share of VAP"]              = fmt_pct(vap_grp[key] / vap_den) if vap_grp and vap_den and key in vap_grp else ""
    row["Share of CVAP"]             = fmt_pct(cvap_grp[key] / cvap_den) if cvap_grp and cvap_den and key in cvap_grp else ""

    rows.append(row)

rows.append({
    "Racial group": "Total count",
    "Share of total population": f"{total_count:,}" if total_count is not None else "",
    "Share of VAP": f"{vap_count:,}" if vap_count is not None else "",
    "Share of CVAP": f"{cvap_count:,}",
})

table_df = pd.DataFrame(rows, columns=["Racial group", "Share of total population", "Share of VAP", "Share of CVAP"])
table_df
